# Ablation - PSG only (no CAD)
PromptSpatialGate in the encoder, with a plain skip decoder without prompt-conditioned attention (PromptAttention not used).

In [1]:
import os, sys, shutil
import gdown

BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
os.chdir(BASE)

REPO = 'https://github.com/ThongLuc2k3/PGA_Unet2D.git'
REPO_ROOT = f'{BASE}/PGA_Unet2D'
PGA_ROOT = f'{REPO_ROOT}/Source/Prompt-Guided-XRay-Segmentation'
DATASET_NAME = 'dataset_FracAtlas'
DATASET_ID = '1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv'
DS_ZIP = f'{BASE}/{DATASET_NAME}.zip'
DS_PATH = f'{PGA_ROOT}/{DATASET_NAME}'

if os.path.exists(REPO_ROOT):
    shutil.rmtree(REPO_ROOT)
!git clone -q --branch main --single-branch {REPO} {REPO_ROOT}
if PGA_ROOT not in sys.path:
    sys.path.insert(0, PGA_ROOT)

if not os.path.exists(DS_ZIP):
    gdown.download(f'https://drive.google.com/uc?id={DATASET_ID}', DS_ZIP, quiet=False)
if os.path.exists(DS_PATH):
    shutil.rmtree(DS_PATH)
!unzip -oq {DS_ZIP} -d {PGA_ROOT}/

os.makedirs(f'{PGA_ROOT}/checkpoints', exist_ok=True)
os.chdir(PGA_ROOT)
!pip install -q tqdm opencv-python timm scipy gdown
print(f'✅ Setup completed | base={BASE} | dataset={DATASET_NAME}')


Downloading...
From (original): https://drive.google.com/uc?id=1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv
From (redirected): https://drive.google.com/uc?id=1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv&confirm=t&uuid=e70a9143-bec0-4bc0-98b2-a662c6617a24
To: /kaggle/working/dataset_FracAtlas.zip
100%|██████████| 231M/231M [00:05<00:00, 42.6MB/s]


✅ Setup completed | base=/kaggle/working | dataset=dataset_FracAtlas


## Ablation training via the shared `train.py`

**Configuration: PSG only (no CAD).** PSG encoder gate only, plain decoder, no CAD. The `psg_only` network class has no QualityHead branch, so `USE_QUALITY_HEAD=0`.

Every ablation configuration is trained through the same `train.py` as the full PGA-UNet, so batch size, optimizer, scheduler, gradient clipping, the 150-epoch / patience-15 early stopping, and **image-level merged `center_shift` validation for checkpoint selection** are identical across the study. Repeat the training cell with `SEED = 1`, `2`, `3` for the multi-seed ablation; `RUN_TAG` keeps the per-seed checkpoints from overwriting each other.

A quick image-level metrics table on the held-out test set is printed by the last cell. For the full qualitative panels and per-image CSVs, run the matching notebook under `Source/File_Test/fracatlas/Ablation/`.

In [2]:
# Repeat with SEED = 1, 2, 3 ... for the multi-seed ablation.
SEED = 22120196
RUN_TAG = f"seed{SEED}"

import os, glob
os.chdir(PGA_ROOT)
os.environ.update({
    "PROMPT_DATASET_ROOT": "dataset_FracAtlas",
    "PROMPT_IMG_SIZE": "512",
    "PROMPT_MODE": "center_mixed",
    "PROMPT_SCALE_FACTOR": "3.0",
    "PROMPT_SHIFT_RATIO": "0.5",
    "PROMPT_MIXED_SHIFT_PROB": "0.8",
    "PROMPT_EPOCHS": "150",
    "PROMPT_SEED": str(SEED),
    "RUN_TAG": RUN_TAG,
    "PROMPT_VARIANT": "psg_only",
    "USE_ENCODER_PROMPT": "1",
    "BINARY_PROMPT": "0",
    "USE_QUALITY_HEAD": "0",
})

!python train.py

print(sorted(glob.glob(f"checkpoints/pga_unet_center_mixed_x3_shift05_psg_only_512_{RUN_TAG}_best.pth")))


TrainPrompt: center_mixed | Device: cuda | Variant: psg_only | EncoderPrompt: True | BinaryPrompt: False | ImgSize: 512 | DatasetRoot: dataset_FracAtlas | Seed: 22120196 | RunTag: seed22120196 | ScaleFactor: 3.0 | ShiftRatio: 0.5 | MixedShiftProb: 0.8
Epoch 1/150 [Train]: 100%|███████| 183/183 [00:36<00:00,  4.97it/s, loss=1.3873]
🥇 [BEST] Epoch   1 | T_Loss: 1.4673 | [CENTER_] Dice:0.0002 IoU:0.0001 CBL:0.0483 | [CENTER_] Dice:0.0002 IoU:0.0001 CBL:0.0484 | LR:1.0e-04
Epoch 2/150 [Train]: 100%|███████| 183/183 [00:36<00:00,  4.97it/s, loss=1.3410]
Epoch   2 | T_Loss: 1.3640 | [CENTER_] Dice:0.0000 IoU:0.0000 CBL:0.0144 | [CENTER_] Dice:0.0000 IoU:0.0000 CBL:0.0144 | LR:1.0e-04
Epoch 3/150 [Train]: 100%|███████| 183/183 [00:38<00:00,  4.70it/s, loss=1.3089]
Epoch   3 | T_Loss: 1.3232 | [CENTER_] Dice:0.0000 IoU:0.0000 CBL:0.0106 | [CENTER_] Dice:0.0000 IoU:0.0000 CBL:0.0106 | LR:1.0e-04
Epoch 4/150 [Train]: 100%|███████| 183/183 [00:38<00:00,  4.72it/s, loss=1.2682]
Epoch   4 | T_Loss:

In [3]:
# -- Quick test on the held-out set: image-level merged, 2 prompt scenarios --
# Loads the checkpoint train.py just wrote in this session (no Drive download).
# Full qualitative panels + per-image CSVs live in Source/File_Test/*/Ablation/.
import os, sys, csv
import numpy as np, torch
from collections import OrderedDict
from torch.utils.data import DataLoader
from tqdm import tqdm
from scipy.ndimage import binary_erosion, distance_transform_edt

os.chdir(PGA_ROOT)
if PGA_ROOT not in sys.path:
    sys.path.insert(0, PGA_ROOT)
from dataset import PromptSegmentationDataset

IMG_SIZE, SCALE_FACTOR, SHIFT_RATIO = 512, 3.0, 0.5
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DS_ROOT = f"{PGA_ROOT}/dataset_FracAtlas"
CKPT    = f"checkpoints/pga_unet_center_mixed_x3_shift05_psg_only_512_{RUN_TAG}_best.pth"

VARIANT, ENC, QH, BINP = "psg_only", 1, 0, 0
if VARIANT == "psg_only":
    from models.networks.prompt_unet_psg_only import PGA_UNet
    model = PGA_UNet(in_channels=1, n_classes=1)
elif VARIANT == "psg_attention":
    from models.networks.prompt_unet_psg_attention import PGA_UNet
    model = PGA_UNet(in_channels=1, n_classes=1)
else:
    from models.networks.prompt_unet_2D import PGA_UNet
    model = PGA_UNet(in_channels=1, n_classes=1,
                     use_encoder_prompt=bool(ENC), use_quality_head=bool(QH))
model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=True))
model.to(DEVICE).eval()
print("loaded", CKPT)


def _hd95(p, g):
    p, g = p.astype(bool), g.astype(bool)
    if not p.any() and not g.any(): return 0.0
    if not p.any() or not g.any():  return float(IMG_SIZE)
    pe, ge = p ^ binary_erosion(p), g ^ binary_erosion(g)
    d1, d2 = distance_transform_edt(~ge)[pe], distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))


def _metrics(prob, gt):
    pm, gm, eps = (prob > 0.5), (gt > 0.5), 1e-6
    tp, fp, fn = float((pm & gm).sum()), float((pm & ~gm).sum()), float((~pm & gm).sum())
    cbl = 0.0
    if gm.any() and pm.any():
        ys, xs = np.where(gm); yp, xp = np.where(pm)
        diag = np.hypot(ys.max() - ys.min(), xs.max() - xs.min()) + eps
        cbl = float(np.clip(1 - np.hypot(xp.mean() - xs.mean(), yp.mean() - ys.mean()) / diag, 0, 1))
    return dict(dice=(2*tp+eps)/(2*tp+fp+fn+eps), iou=(tp+eps)/(tp+fp+fn+eps),
                pre=tp/(tp+fp+eps), rec=(tp+eps)/(tp+fn+eps), hd95=_hd95(pm, gm), cbl=cbl)


os.makedirs("results", exist_ok=True)
per_image, rows = [], []
for mode in ("center_zoom", "center_shift"):
    ds = PromptSegmentationDataset(f"{DS_ROOT}/test/images", f"{DS_ROOT}/test/annotations",
                                   img_size=IMG_SIZE, is_train=False, prompt_mode=mode,
                                   scale_factor=SCALE_FACTOR, shift_ratio=SHIFT_RATIO,
                                   binary_prompt=bool(BINP))
    groups = OrderedDict()
    with torch.no_grad():
        for i, (img, msk, prm) in enumerate(tqdm(DataLoader(ds, batch_size=1, num_workers=2), desc=mode)):
            prob = torch.sigmoid(model(img.to(DEVICE), prm.to(DEVICE)))[0, 0].cpu().numpy()
            gt   = msk[0, 0].numpy()
            name = ds.all_samples[i][0]
            if name not in groups:
                groups[name] = [prob.copy(), gt.copy()]
            else:
                np.maximum(groups[name][0], prob, out=groups[name][0])
                np.maximum(groups[name][1], gt,   out=groups[name][1])
    recs = []
    for name, (p, g) in groups.items():
        m = _metrics(p, g); recs.append(m)
        per_image.append([mode, name] + [m[k] for k in ("dice", "iou", "pre", "rec", "hd95", "cbl")])
    avg = {k: float(np.mean([r[k] for r in recs])) for k in recs[0]}
    rows.append((mode, len(recs), avg))

print(f"\n{'scenario':<14}{'N':>5}{'Dice':>9}{'IoU':>9}{'Prec':>9}{'Rec':>9}{'HD95':>9}{'CBL':>9}")
for mode, n, a in rows:
    print(f"{mode:<14}{n:>5}{a['dice']:>9.4f}{a['iou']:>9.4f}{a['pre']:>9.4f}{a['rec']:>9.4f}{a['hd95']:>9.2f}{a['cbl']:>9.4f}")

csv_path = f"results/ablation_psg_only_{RUN_TAG}_per_image.csv"
with open(csv_path, "w", newline="") as fh:
    w = csv.writer(fh); w.writerow(["scenario", "img_name", "dice", "iou", "pre", "rec", "hd95", "cbl"])
    w.writerows(per_image)
print("saved", csv_path)


loaded checkpoints/pga_unet_center_mixed_x3_shift05_psg_only_512_seed22120196_best.pth


center_shift: 100%|██████████| 92/92 [00:02<00:00, 34.34it/s]



scenario          N     Dice      IoU     Prec      Rec     HD95      CBL
center_zoom      72   0.6350   0.4793   0.5317   0.8790    14.51   0.8940
center_shift     72   0.6111   0.4594   0.5135   0.8354    14.83   0.8567
saved results/ablation_psg_only_seed22120196_per_image.csv
